In [16]:
import sys
!{sys.executable} -m pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [5]:

import pandas as pd
import datetime as dt

# Carichiamo il file che Python vede direttamente nella cartella corrente
try:
    # Se hai installato openpyxl, usiamo read_excel
    df = pd.read_excel('Online Retail.xlsx')
    print("✅ File caricato con successo!")
    display(df.head())
except Exception as e:
    print(f"❌ Errore: {e}")

✅ File caricato con successo!


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
import pandas as pd
import datetime as dt

# 1. Caricamento del file Excel
# Usiamo read_excel perché il tuo file ha estensione .xlsx
df = pd.read_excel('Online Retail.xlsx')

print("--- AUDIT INIZIALE ---")
print(f"Righe totali: {len(df)}")

# 2. Pulizia Dati (Seguendo i "Data Cleaning Essentials")
# Rimuoviamo i CustomerID mancanti (non possiamo segmentare clienti anonimi)
df = df.dropna(subset=['CustomerID'])

# Rimuoviamo quantità e prezzi negativi (resi e correzioni)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Calcoliamo il Totale per riga
df['TotalSum'] = df['Quantity'] * df['UnitPrice']

print(f"Righe dopo la pulizia: {len(df)}")
display(df.head())

--- AUDIT INIZIALE ---
Righe totali: 541909
Righe dopo la pulizia: 397884


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSum
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [8]:
# 1. Calcolo dei valori RFM reali
# Definiamo la data di riferimento (un giorno dopo l'ultimo acquisto nel dataset)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'count',                                   # Frequency
    'TotalSum': 'sum'                                       # Monetary
})

# Rinominiamo le colonne
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSum': 'Monetary'
}, inplace=True)

# 2. Assegnazione dei punteggi da 1 a 5 (Scoring)
# Per la Recency, il numero di giorni PIÙ BASSO è il migliore, quindi invertiamo le etichette
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])

# Per Frequency e Monetary, il numero PIÙ ALTO è il migliore
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

# Creiamo un punteggio totale (somma dei tre)
rfm['RFM_Score'] = rfm[['R_score', 'F_score', 'M_score']].sum(axis=1)

print("--- SEGMENTAZIONE RFM COMPLETATA ---")
display(rfm.head())

--- SEGMENTAZIONE RFM COMPLETATA ---


,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
CustomerID,,,,,,,
12346.0,326,1,77183.60,1,1,5,7
12347.0,2,182,4310.00,5,5,5,15
12348.0,75,31,1797.24,2,3,4,9
12349.0,19,73,1757.55,4,4,4,12
12350.0,310,17,334.40,1,2,2,5
